In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import pathlib
from pathlib import Path

First, I wrote two functions to compile all the calibration files into a dictionary for the corresponding observation.

In [ ]:
def load_data(file):
"""
Takes input the fits file, and return the data,
primary header, and data header

Parameters:
    -----------
    

    Returns:
    -----------

"""

    with fits.open(file) as hdul:
        pheader = hdul[0].header

        # For the already calibrated files, only need to return
        # the data
        if str(file).endswith("aligned.fits"):
            data = hdul[1].data
            return data
        # 2507 is structured differently, so for this observation
        # the data is from the the first header
        elif '2507' in str(file):
            data = hdul[0].data
        # raw and blev data is in header 1
        elif str(file).endswith("raw.fits") or str(file).endswith("blevfile.fits"):
            data = hdul[1].data
        else:
            data = hdul[0].data
        dheader = hdul[1].header
    return data, pheader, dheader

In [ ]:
def load_file(filelist):
"""
Take the list of files, and build the dictionary for easy access

Parameters:
    -----------
    

    Returns:
    -----------

"""

    file_dict = {}
    
    for file in filelist:

        # For each file in the file list, we call the load_data function
        data, pheader, dheader = load_data(file)
        file_dict[str(file)] = (data, pheader, dheader)
    return file_dict

I then wrote two functions to handle the blev file

In [ ]:
def blev_parsing(data):
"""
Takes the blev data in, and returns the mean values
of the even and odd overscan regions

Parameters:
    -----------
    

    Returns:
    -----------

"""
    # Define the area of interest of the data
    area = data[9:790, 8:14]
    oddv = area[:, 0::2]
    evenv = area[:, 1::2]
    return np.mean(oddv), np.mean(evenv)

In [ ]:
def apply_bias(raw, blev):
"""
Takes the raw and blev data, and return the adjusted
data 

Parameters:
    -----------
    

    Returns:
    -----------

"""

    # Make a copy of the raw data
    corr_data = raw.astype('float64')
    
    # Multi chip case
    if raw.ndim == 3 and blev.ndim == 3 and blev.shape[-1] == 14 and raw.shape[0] == blev.shape[0]:
        for c in range(raw.shape[0]):

            # Load mean odds and evens from blev_parsing
            mean_o, mean_e = blev_parsing(blev[c])

            # Subtract the means from the data
            corr_data[c, :, 0::2] -= mean_o
            corr_data[c, :, 1::2] -= mean_e
        return corr_data

    # Single chip case  
    mean_o, mean_e = blev_parsing(blev)
    corr_data[:, 0::2] -= mean_o
    corr_data[:, 1::2] -= mean_e
    return corr_data

After the bias function, I wrote a simple function to run all the calibration steps.

In [ ]:
def calibration(dpath):
"""
Takes the paths to the fits files, and return the final_cal
calibrated file and exposure time

Parameters:
    -----------
    

    Returns:
    -----------

"""

    # Create the corresponding dictionary with the load_file function
    files = load_file(dpath)
    for filename, file_tuple in files.items():

        # Cycle through the files
        if 'raw' in filename:
            raw_data = file_tuple[0]
            exptime = file_tuple[1]['EXPTIME']
        elif 'bias' in filename:
            bias_data = file_tuple[0]
        # Decided to ignore the mask data
        #elif 'mask' in filename:
        #    mask_data = file_tuple[0]
        elif 'blev' in filename:
            blev_data = file_tuple[0]
        elif 'dark' in filename:
            dark_data = file_tuple[0]
        elif 'flat' in filename:
            flat_data = file_tuple[0]

    # Apply all the calibration steps as described in the brief
    corr_blev = apply_bias(raw_data, blev_data)
    corr_bias = corr_blev - bias_data
    # Scaling the dark_data for the exposure time
    dark_scaled = dark_data * exptime
    corr_dark = corr_bias - dark_scaled
    corr_flat = corr_dark * flat_data
    #bad_pixels = mask_data > 0
    #corr_flat[bad_pixels] = np.nan
    sky_bkg = np.nanmedian(corr_flat)
    final_cal = corr_flat - sky_bkg
    return final_cal, exptime

After the calibration, I then wrote a function to apply the photometry.

In [ ]:
def photom(aligned_file,exptime, photflam = 1.115396e-16,
           photzpt = -21.1, zp_corr = -0.03288031):
"""
Takes the calibrated file and exposure time, and returns
flux and magnitude

Parameters:
    -----------
    

    Returns:
    -----------

"""

    # Calculate the rate for flux and magnitude
    rate = aligned_file / exptime
    flux = rate * photflam
    mag = (-2.5 * np.log10(rate * photflam) + photzpt) + zp_corr
    return flux, mag

Finally, I wrote twofunctions to check the calibrated files from the pipeline with the aligned files, as well as the photometry comparison.

In [ ]:
def calib_check(og, my, title, outdir):
"""
Takes the aligned and calibrated files, the plot title
and directory, and save the comparative plots in the
specified directory 

Parameters:
    -----------
    

    Returns:
    -----------

"""

    # Assign the output directory
    outdir = Path(outdir)
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(10, 5))

    # Plot the pipeline-calibrated and aligned images next to each other
    for ax, img, name in ((ax1, og, "Aligned file"), (ax2, my, "Calibrated file")):
        vmin, vmax = np.nanpercentile(img, [1, 99])
        ax.imshow(img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_title(name)
    fig.suptitle(title)
    plt.show()

    # Save the plot to the directory
    fig.savefig(outdir / f"Calib_check_{title}.jpg", dpi=300, bbox_inches='tight')

In [ ]:
def flux_check(og, my, title, outdir):
"""
Takes the aligned and calibrated files, the plot title
and directory, and save the comparative flux plots in the
specified directory 

Parameters:
    -----------
    

    Returns:
    -----------

"""

    # Assign the output directory
    outdir = Path(outdir)
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(11, 5), constrained_layout=True)

    # Plot the pipeline-calibrated and aligned images next to each other
    for ax, img, name in ((ax1, og, "Aligned file"), (ax2, my, "Calibrated file")):
        vmin, vmax = np.nanpercentile(img, [1, 99])
        im = ax.imshow(img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_title(name)

        # Add the colorbar for the flux 
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(r"Flux [erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$]")
    fig.suptitle(title)
    plt.show()

    # Save the plot to the directory
    fig.savefig(outdir / f"Flux_check_{title}.jpg", dpi=300, bbox_inches='tight')

Once all the functions were written down, I proceeded to call them in order

In [ ]:
# Build the calibration file paths
here = pathlib.Path('.')
ubai2505m = list(here.rglob('**/ubai2505m/**/*.fits'))
ubai2507m = list(here.rglob('**/ubai2507m/**/*.fits'))
ubai2508m = list(here.rglob('**/ubai2508m/**/*.fits'))